In [ ]:
from deepface import DeepFace  # Qué: importa la librería DeepFace. Cómo: envuelve varios modelos de deep learning (VGG-Face, Facenet, etc.) para detección, verificación y búsqueda de rostros. Por qué: evita implementar reconocimiento facial desde cero; se usa aquí su función `find` para comparar contra una base de datos de fotos.
import cv2  # Qué: importa OpenCV. Por qué: se usa para capturar video de la webcam y mostrar/anotar los cuadros con el nombre reconocido.
import pyttsx3  # Qué: importa la librería de texto a voz (Text-To-Speech) offline. Por qué: permite que el sistema "salude" en voz alta a la persona reconocida, sin depender de un servicio en la nube.

In [ ]:
def reconocimiento(frame):  # Qué: función que intenta identificar a la persona presente en `frame`. Por qué: encapsula toda la lógica de búsqueda facial + manejo de errores en un único punto reutilizable dentro del loop principal.
    try:
        recognition = DeepFace.find(frame,db_path='../db',model_name='VGG-Face',silent=True)  # Qué: busca el rostro de `frame` dentro de la base de datos de imágenes en `../db`. Cómo: DeepFace detecta el rostro en `frame`, genera su "embedding" (vector numérico que representa la cara) con el modelo VGG-Face, y lo compara contra los embeddings de todas las fotos en `db_path` (organizadas en subcarpetas por persona) devolviendo un DataFrame ordenado por similitud; `silent=True` suprime los logs de progreso. Por qué: reutiliza fotos ya etiquetadas por carpeta en vez de entrenar un clasificador propio.
        print(recognition[0]['identity'][0])  # Qué: imprime en consola la ruta de la imagen más parecida encontrada (la primera fila del primer DataFrame de resultados). Por qué: sirve como log de depuración para ver qué coincidencia exacta se usó.

        recognition2 = recognition[0]['identity'][0]  # Qué: guarda esa misma ruta de la mejor coincidencia. Por qué: se necesita como string para extraer el nombre de la persona en el siguiente paso.
        nombre = recognition2.split('\\')[1].split('/')[0]  # Qué: extrae el nombre de la persona a partir de la ruta del archivo. Cómo: la ruta tiene forma `../db\\NombrePersona\\foto.jpg` (Windows usa `\`); `split('\\')[1]` toma el segundo segmento (el nombre de la subcarpeta) y `split('/')[0]` es una limpieza extra por si aparecieran barras normales mezcladas. Por qué: el nombre de la persona está codificado como el nombre de la subcarpeta dentro de `db`, no como un campo separado.
        return nombre  # Qué: devuelve el nombre identificado al llamador. Por qué: es el resultado útil de la función cuando el reconocimiento tuvo éxito.
    except ValueError:  # Qué: captura el error que DeepFace lanza cuando NO detecta ningún rostro en `frame`. Por qué: DeepFace.find falla con ValueError si el detector de caras interno no encuentra una cara válida en la imagen de entrada (p. ej. cuadro vacío o rostro no visible); sin este catch, el loop principal se rompería en cada cuadro sin cara.
        return 'Rostro No Detectado'  # Qué: valor de reemplazo cuando no hay cara. Por qué: mensaje consistente que el resto del programa puede comparar/mostrar sin necesitar lógica adicional.
    except KeyError:  # Qué: captura el error que ocurre cuando DeepFace SÍ detecta un rostro pero no encuentra ninguna coincidencia en la base de datos. Por qué: en ese caso el DataFrame de resultados viene vacío y acceder a `['identity'][0]` lanza KeyError/IndexError sobre un índice inexistente; se distingue de ValueError porque es un fallo distinto (cara detectada pero desconocida, vs. ninguna cara detectada).
        return 'Rostro No Detectado'  # Qué: mismo mensaje de fallback. Por qué: desde la perspectiva del usuario final, "no reconocido" y "no detectado" requieren la misma reacción (no saludar).

In [ ]:
engine = pyttsx3.init()  # Qué: inicializa el motor de texto a voz. Cómo: detecta y conecta con el motor TTS disponible en el sistema operativo (SAPI5 en Windows). Por qué: se crea una sola vez fuera del loop porque instanciar el motor es costoso y no hace falta repetirlo en cada cuadro.

engine.setProperty('rate',125)  # Qué: configura la velocidad de habla del motor TTS. Cómo: `rate` son palabras por minuto aproximadas (el valor por defecto suele rondar 200). Por qué: 125 reduce la velocidad para que el saludo se entienda con más claridad.

In [ ]:
def saludar(mensaje):  # Qué: función que reproduce por voz un saludo a la persona identificada. Por qué: separa la lógica de "hablar" de la lógica de "reconocer", manteniendo cada función con una sola responsabilidad.
    if mensaje != 'Rostro No Detectado':  # Qué: solo saluda si hubo un reconocimiento válido. Por qué: evita que el sistema hable constantemente cuando no hay nadie o no se reconoce a la persona frente a cámara.
        engine.say('hola '+ mensaje)  # Qué: encola el texto "hola <nombre>" para ser sintetizado a voz. Cómo: `say` no reproduce de inmediato, solo agrega el texto a la cola interna del motor. Por qué: DeepFace ya devolvió el nombre de la persona reconocida, y se arma un saludo simple con ese dato.
        engine.runAndWait()  # Qué: procesa la cola de texto y reproduce el audio de forma bloqueante. Cómo: bloquea la ejecución hasta que termina de hablar. Por qué: sin esta llamada, `say()` no produce ningún sonido (queda solo en cola).

In [ ]:
vid = cv2.VideoCapture(0)  # Qué: abre la webcam por defecto (índice 0). Por qué: fuente de video en vivo sobre la que se ejecuta el reconocimiento facial.
cv2.namedWindow('Reconocimiento Facial',cv2.WINDOW_NORMAL)  # Qué: crea explícitamente la ventana antes de usar imshow. Cómo: el flag WINDOW_NORMAL la hace redimensionable por el usuario (a diferencia de WINDOW_AUTOSIZE, que la fija al tamaño de la imagen). Por qué: da flexibilidad para ajustar el tamaño de la ventana de video en pantalla.

mensajeAnterior = ''  # Qué: variable que guarda el último mensaje/nombre reconocido en el ciclo anterior. Por qué: se usa como memoria de un cuadro a otro para detectar cambios de identidad.

while True:  # Qué: procesa la webcam en vivo, cuadro por cuadro, hasta que el usuario salga.
    
    ret,frame = vid.read()  # Qué: captura el siguiente cuadro de la webcam. Cómo: `ret` indica éxito de lectura, `frame` es la imagen BGR actual.
  
    mensaje = reconocimiento(frame)  # Qué: ejecuta el reconocimiento facial sobre el cuadro actual. Por qué: DeepFace.find es costoso (corre un modelo de deep learning), pero se llama en cada frame en esta versión simple del notebook (la práctica optimiza esto con un throttle).

    if mensaje != mensajeAnterior:  # Qué: compara el resultado actual contra el del cuadro anterior. Por qué: evita repetir el saludo por voz en cada frame mientras la misma persona sigue frente a cámara; solo se saluda de nuevo si el nombre reconocido cambió (p. ej. entra otra persona, o pasa de "no detectado" a un nombre).
        saludar(mensaje) 

    cv2.putText(frame,mensaje,(0,115),cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0))  # Qué: dibuja el nombre reconocido (o "Rostro No Detectado") sobre el cuadro. Cómo: fuente FONT_HERSHEY_SIMPLEX, escala 1, color verde (BGR: 0,255,0), anclado en la posición (0,115). Por qué: da feedback visual en pantalla de qué identificó el sistema, además del saludo por voz.

    cv2.imshow('Reconocimiento Facial',frame)  # Qué: muestra el cuadro anotado en la ventana creada arriba. Por qué: es la salida visual en vivo del reconocimiento.
    
    if cv2.waitKey(1) == ord('q'):  # Qué: espera 1 ms por una tecla y compara con 'q'. Por qué: permite salir del loop de forma controlada.
        break  # Qué: corta el bucle infinito. Por qué: única salida controlada del loop de reconocimiento.
    mensajeAnterior = mensaje  # Qué: actualiza la memoria del "último mensaje" con el mensaje de este cuadro. Por qué: debe ejecutarse al final de cada iteración para que la próxima vuelta del loop pueda comparar correctamente y detectar si hubo cambio de identidad.

vid.release()  # Qué: libera el dispositivo de cámara. Por qué: evita bloquear la webcam para otros procesos.
cv2.destroyAllWindows()  # Qué: cierra todas las ventanas abiertas de OpenCV. Por qué: limpieza de recursos de GUI al terminar.

## 🧪 Práctica
Reforzá lo aprendido en este módulo resolviendo los ejercicios guiados en [`practicas/8_practica.ipynb`](../practicas/8_practica.ipynb).